## Part 2 — Masked CD-1 training

Movie-level mask `M2` is expanded to `n_movies×K` **inside each batch** for B1  
(avoids a dense 2.6 GB `M1` in RAM).  
B1 is loaded → trained → freed, then B2 is trained.


## Part 0 — Setup

`X_DTYPE`: prefer `float32` (X1 ≈ 2.6 GB).  
On ≤8 GB machines: do **not** materialize dense `M1` (another ~2.6 GB).  
Part 1–2 load/train **one channel at a time** and expand the movie mask to K only inside each batch.


In [8]:
from pathlib import Path
import shutil
import time

import numpy as np

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

proc = root / "data" / "processed"
assert proc.exists(), f"Missing {proc}"

N_HIDDEN = 128
LR = 0.01
BATCH_SIZE = 500
EPOCHS = 50
INIT_SEED = 42
K = 10
X_DTYPE = np.float32        # np.float64 if RAM allows

path_b1 = proc / "channelB1_softmax.npy"
path_b2 = proc / "channelB2_personality.npy"
path_mask = proc / "mask.npy"
for p in (path_b1, path_b2, path_mask):
    assert p.exists(), f"Missing {p} — run notebooks 09b / 12b first."

ch1_meta = np.load(path_b1, mmap_mode="r")
ch2_meta = np.load(path_b2, mmap_mode="r")
mask_meta = np.load(path_mask, mmap_mode="r")
n_users, n_movies, k_b1 = ch1_meta.shape
assert k_b1 == K
assert ch2_meta.shape == (n_users, n_movies, 1)
assert mask_meta.shape == (n_users, n_movies)

n_vis_b1 = n_movies * K
n_vis_b2 = n_movies
bytes_x1 = n_users * n_vis_b1 * np.dtype(X_DTYPE).itemsize
bytes_x2 = n_users * n_vis_b2 * np.dtype(X_DTYPE).itemsize
bytes_w1 = n_vis_b1 * N_HIDDEN * 8
free = shutil.disk_usage(proc).free

print(f"Project root: {root}")
print(f"Cohort: n_users={n_users:,}, n_movies={n_movies:,}, K={K}")
print(f"\n=== Memory / disk estimate ===")
print(f"  X1 ({X_DTYPE.__name__}):     {bytes_x1 / 1e9:.2f} GB   shape=({n_users}, {n_vis_b1})")
print(f"  X2 ({X_DTYPE.__name__}):     {bytes_x2 / 1e9:.2f} GB   shape=({n_users}, {n_vis_b2})")
print(f"  W1 (float64, RAM): {bytes_w1 / 1e6:.1f} MB")
print(f"  Disk free:         {free / 1e9:.2f} GB")
print(f"\nHyperparams: N_HIDDEN={N_HIDDEN}, LR={LR}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, SEED={INIT_SEED}")
print("No deltaW logging — only final W / b_h / mse_log will be saved.")
print("Proceed to Part 1.")

del ch1_meta, ch2_meta, mask_meta


Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Cohort: n_users=5,000, n_movies=13,129, K=10

=== Memory / disk estimate ===
  X1 (float32):     2.63 GB   shape=(5000, 131290)
  X2 (float32):     0.26 GB   shape=(5000, 13129)
  W1 (float64, RAM): 134.4 MB
  Disk free:         3.62 GB

Hyperparams: N_HIDDEN=128, LR=0.01, BATCH_SIZE=500, EPOCHS=50, SEED=42
No deltaW logging — only final W / b_h / mse_log will be saved.
Proceed to Part 1.


## Part 1 — Load data / flatten / masks

In [9]:
import gc

print("Loading mask + channelB2 first (small); channelB1 loaded later for training only …")
mask = np.load(path_mask)  # int8 (n_users, n_movies) ≈ 66 MB
channelB2 = np.load(path_b2)  # float32 (n_users, n_movies, 1) ≈ 0.26 GB

n_users, n_movies = mask.shape
assert channelB2.shape == (n_users, n_movies, 1)

X2 = channelB2.reshape(n_users, -1).astype(X_DTYPE, copy=False)
del channelB2
M2 = mask.astype(X_DTYPE, copy=False)  # (n_users, n_movies) — NOT repeated ×K

# Peek B1 shape without loading full tensor
b1_mm = np.load(path_b1, mmap_mode="r")
assert b1_mm.shape == (n_users, n_movies, K)
print(f"B1 on disk: {b1_mm.shape} (will load into X1 only when training RBM_B1)")
del b1_mm

print(f"X2 {X2.shape} {X2.dtype}")
print(f"M2 {M2.shape} {M2.dtype}  (movie-level mask; K-expand happens per batch)")
print(f"mask {mask.shape} {mask.dtype}")

nz2 = M2.sum(axis=1)
print(f"✓ movie-level mask ready")
print(f"  train movies/user: min={int(nz2.min())}, median={float(np.median(nz2)):.0f}, max={int(nz2.max())}")
print("Peak RAM tip: X1 (~2.6GB) is loaded in Part 2, then freed before/after as needed.")
gc.collect()


Loading mask + channelB2 first (small); channelB1 loaded later for training only …
B1 on disk: (5000, 13129, 10) (will load into X1 only when training RBM_B1)
X2 (5000, 13129) float32
M2 (5000, 13129) float32  (movie-level mask; K-expand happens per batch)
mask (5000, 13129) int8
✓ movie-level mask ready
  train movies/user: min=495, median=696, max=6032
Peak RAM tip: X1 (~2.6GB) is loaded in Part 2, then freed before/after as needed.


66

## Part 2 — Masked CD-1 training

After negative-phase `v_recon_prob` / `v_recon`, multiply by the batch mask so test/unrated visibles contribute no gradient.

No per-epoch ΔW tensor is kept — only final weights and MSE at epochs 1, 25, 50.


In [10]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))


def expand_mask_batch(m_movies, k):
    """(batch, n_movies) -> (batch, n_movies*k) without keeping a global M1."""
    return np.repeat(m_movies, k, axis=1)


def reconstruction_mse(X, M_movies, W, b_h, k_expand, chunk=250):
    """Masked MSE in row-chunks — never allocates a full (n_users, n_visible) recon."""
    n = X.shape[0]
    num = 0.0
    den = 0.0
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n)
        x = np.asarray(X[i0:i1])
        m = M_movies[i0:i1]
        if k_expand > 1:
            m = expand_mask_batch(m, k_expand)
        m = np.asarray(m, dtype=x.dtype)
        h_prob = sigmoid(x @ W + b_h)
        v_prob = sigmoid(h_prob @ W.T)
        diff2 = (x - v_prob) ** 2
        num += float((diff2 * m).sum())
        den += float(m.sum())
    return num / den if den > 0 else float("nan")


def train_rbm(X, M_movies, n_hidden, seed, channel_name, k_expand=1):
    """Masked CD-1. Expand movie mask ×k per batch when k_expand=K (no dense M1)."""
    rng = np.random.default_rng(seed)
    n_samples, n_visible = X.shape
    assert M_movies.shape[0] == n_samples

    W = rng.normal(0.0, 0.01, size=(n_visible, n_hidden)).astype(np.float64)
    b_h = np.zeros(n_hidden, dtype=np.float64)

    mse_log = {}
    t0_all = time.perf_counter()

    print(f"\n=== Training {channel_name} ===", flush=True)
    print(f"  samples={n_samples}, visible={n_visible}, hidden={n_hidden}", flush=True)
    print(f"  lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}, init_seed={seed}, k_expand={k_expand}", flush=True)

    for epoch in range(EPOCHS):
        t0 = time.perf_counter()
        print(f"  … epoch {epoch + 1}/{EPOCHS} training", flush=True)
        epoch_delta = np.zeros_like(W)
        order = rng.permutation(n_samples)

        for start in range(0, n_samples, BATCH_SIZE):
            idx = order[start : start + BATCH_SIZE]
            v_data = np.asarray(X[idx])  # keep X dtype (float32)
            m_batch = M_movies[idx]
            if k_expand > 1:
                m_batch = expand_mask_batch(m_batch, k_expand)
            m_batch = np.asarray(m_batch, dtype=X.dtype)
            bs = v_data.shape[0]

            h_prob = sigmoid(v_data @ W + b_h)
            h_data = (rng.random(h_prob.shape) < h_prob).astype(np.float64)

            v_recon_prob = sigmoid(h_data @ W.T) * m_batch
            v_recon = (rng.random(v_recon_prob.shape) < v_recon_prob).astype(np.float64) * m_batch

            h_recon_prob = sigmoid(v_recon @ W + b_h)
            h_recon = (rng.random(h_recon_prob.shape) < h_recon_prob).astype(np.float64)

            pos = v_data.T @ h_data
            neg = v_recon.T @ h_recon
            epoch_delta += LR * (pos - neg) / bs
            b_h += LR * (h_prob.mean(axis=0) - h_recon_prob.mean(axis=0))

        W += epoch_delta
        dt = time.perf_counter() - t0

        ep = epoch + 1
        if ep in (1, 25, EPOCHS):
            print(f"  … computing masked MSE (chunked) …", flush=True)
            mse = reconstruction_mse(X, M_movies, W, b_h, k_expand)
            mse_log[ep] = mse
            print(f"  Epoch {ep:02d} | masked recon MSE: {mse:.6f} | epoch wall {dt:.1f}s", flush=True)
        else:
            print(f"  Epoch {ep:02d} | epoch wall {dt:.1f}s", flush=True)

        if ep == 1:
            print(
                f"  ⏱ epoch 1 wall time: {dt:.1f}s  → est. full {EPOCHS} epochs ≈ {dt * EPOCHS / 60:.1f} min",
                flush=True,
            )

    elapsed = time.perf_counter() - t0_all
    print(f"  Done in {elapsed / 60:.1f} min", flush=True)
    return W, b_h, mse_log, elapsed


# --- Train B1: load X1, train, free X1 ---
print("Loading channelB1 into X1 (≈2.6 GB) …", flush=True)
channelB1 = np.load(path_b1)
X1 = channelB1.reshape(n_users, -1)
if X_DTYPE != np.float32:
    X1 = X1.astype(X_DTYPE)
del channelB1
gc.collect()
print(f"X1 {X1.shape} {X1.dtype}", flush=True)

W1, bh1, mse1, t1 = train_rbm(
    X1, M2, N_HIDDEN, seed=INIT_SEED, channel_name="RBMB1 (user rating)", k_expand=K
)

del X1
gc.collect()
print("Freed X1 from RAM.", flush=True)

# --- Train B2: X2 already in RAM (~0.26 GB) ---
W2, bh2, mse2, t2 = train_rbm(
    X2, M2, N_HIDDEN, seed=INIT_SEED, channel_name="RBMB2 (personality)", k_expand=1
)

total_train_sec = t1 + t2
print(f"\nTotal training wall time: {total_train_sec / 60:.1f} min", flush=True)


Loading channelB1 into X1 (≈2.6 GB) …
X1 (5000, 131290) float32

=== Training RBMB1 (user rating) ===
  samples=5000, visible=131290, hidden=128
  lr=0.01, batch=500, epochs=50, init_seed=42, k_expand=10
  … epoch 1/50 training
  … computing masked MSE (chunked) …
  Epoch 01 | masked recon MSE: 0.245642 | epoch wall 29.3s
  ⏱ epoch 1 wall time: 29.3s  → est. full 50 epochs ≈ 24.4 min
  … epoch 2/50 training
  Epoch 02 | epoch wall 24.9s
  … epoch 3/50 training
  Epoch 03 | epoch wall 27.6s
  … epoch 4/50 training
  Epoch 04 | epoch wall 28.2s
  … epoch 5/50 training
  Epoch 05 | epoch wall 28.4s
  … epoch 6/50 training
  Epoch 06 | epoch wall 28.2s
  … epoch 7/50 training
  Epoch 07 | epoch wall 27.4s
  … epoch 8/50 training
  Epoch 08 | epoch wall 26.9s
  … epoch 9/50 training
  Epoch 09 | epoch wall 26.7s
  … epoch 10/50 training
  Epoch 10 | epoch wall 26.8s
  … epoch 11/50 training
  Epoch 11 | epoch wall 26.0s
  … epoch 12/50 training
  Epoch 12 | epoch wall 30.1s
  … epoch 13/50 

## Part 3 — Save `_5k` artifacts

In [11]:
paths = {
    "W1": proc / "rbmB1_weights_5k.npy",
    "W2": proc / "rbmB2_weights_5k.npy",
    "bh1": proc / "rbmB1_bias_hidden_5k.npy",
    "bh2": proc / "rbmB2_bias_hidden_5k.npy",
}

need = W1.nbytes + W2.nbytes + bh1.nbytes + bh2.nbytes
free = shutil.disk_usage(proc).free
print(f"Save check: need ~{need / 1e6:.1f} MB, free {free / 1e9:.2f} GB")
if free < need + 50_000_000:
    raise OSError("Not enough disk to save weight artifacts.")

np.save(paths["W1"], W1)
np.save(paths["W2"], W2)
np.save(paths["bh1"], bh1)
np.save(paths["bh2"], bh2)

meta = {
    "EPOCHS": EPOCHS,
    "N_HIDDEN": N_HIDDEN,
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "INIT_SEED": INIT_SEED,
    "mse1": mse1,
    "mse2": mse2,
    "total_train_sec": total_train_sec,
}
np.save(proc / "rbmB_5k_train_meta.npy", meta)

for k, p in paths.items():
    print(f"Saved {k}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"Saved meta: {proc / 'rbmB_5k_train_meta.npy'}")


Save check: need ~147.9 MB, free 2.52 GB
Saved W1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmB1_weights_5k.npy  (134.4 MB)
Saved W2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmB2_weights_5k.npy  (13.4 MB)
Saved bh1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmB1_bias_hidden_5k.npy  (0.0 MB)
Saved bh2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmB2_bias_hidden_5k.npy  (0.0 MB)
Saved meta: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmB_5k_train_meta.npy


## Part 4 — Verification

In [12]:
def print_W_stats(name, W):
    print(f"{name}: shape={W.shape}")
    print(
        f"  mean={W.mean():.6f}  std={W.std():.6f}  "
        f"min={W.min():.6f}  max={W.max():.6f}"
    )


print("=== Final weights ===")
print_W_stats("W1 (RBMB1)", W1)
print_W_stats("W2 (RBMB2)", W2)

print("\n=== Masked recon MSE ===")
for ep in (1, 25, EPOCHS):
    print(f"  epoch {ep:2d}:  B1={mse1.get(ep, float('nan')):.6f}  B2={mse2.get(ep, float('nan')):.6f}")

# Coarse channel-structure check: mean |W| per hidden unit
w1_col = np.mean(np.abs(W1), axis=0)  # (n_hidden,)
w2_col = np.mean(np.abs(W2), axis=0)
r = float(np.corrcoef(w1_col, w2_col)[0, 1])
print(f"\nPearson r between mean|W| per hidden unit (B1 vs B2): {r:.4f}")
print("(coarse redundancy check — different visible dims, not a matrix Frobenius correlation)")

print(f"\nTotal training time: {total_train_sec / 60:.1f} min ({total_train_sec:.0f}s)")
print("\nAll done.")


=== Final weights ===
W1 (RBMB1): shape=(131290, 128)
  mean=-0.004249  std=0.013697  min=-0.104865  max=0.367044
W2 (RBMB2): shape=(13129, 128)
  mean=-0.054508  std=0.093897  min=-0.692648  max=0.042482

=== Masked recon MSE ===
  epoch  1:  B1=0.245642  B2=0.299260
  epoch 25:  B1=0.161966  B2=0.150547
  epoch 50:  B1=0.147245  B2=0.145279

Pearson r between mean|W| per hidden unit (B1 vs B2): 0.2634
(coarse redundancy check — different visible dims, not a matrix Frobenius correlation)

Total training time: 25.1 min (1505s)

All done.
